In [ ]:
# Locate dataset
import os, yaml

# Find data.yaml anywhere under /kaggle/input
data_yaml = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        data_yaml = os.path.join(root, 'data.yaml')
        break

if data_yaml is None:
    raise FileNotFoundError('data.yaml not found in /kaggle/input')

print(f'Found data.yaml: {data_yaml}')
base = os.path.dirname(data_yaml)
print(f'Dataset base: {base}')
print(f'Contents: {os.listdir(base)}')

# Fix paths to absolute
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')

# Write updated yaml to /kaggle/working so we can modify it
data_yaml_fixed = '/kaggle/working/data.yaml'
with open(data_yaml_fixed, 'w') as f:
    yaml.dump(cfg, f)

print('data.yaml fixed:')
print(cfg)

# Verify splits exist
for split in ['train', 'val', 'test']:
    p = cfg[split]
    count = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f'  {split}: {count} images')


In [ ]:
# Check ultralytics — pre-installed on Kaggle, no pip needed
from ultralytics import YOLO
import torch
import ultralytics
print(f'ultralytics {ultralytics.__version__} ready')
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# Run training — yolo26s on T4
from ultralytics import YOLO
model = YOLO('yolo26s.pt')
results = model.train(
    data=data_yaml_fixed,
    imgsz=640,
    batch=32,
    epochs=100,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/kaggle/working/runs',
    name='anti_uav_run1',
)
print(f'Training complete — results: {results.save_dir}')


In [ ]:
# Archive full run directory
import zipfile, os
runs_dir = '/kaggle/working/runs/anti_uav_run1'
archive_path = '/kaggle/working/anti_uav_run1_full.zip'
print('Archiving...')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/kaggle/working')
            zf.write(filepath, arcname)
            print(f'  {arcname} ({os.path.getsize(filepath)/1e6:.1f} MB)')
print(f'Done: {archive_path} ({os.path.getsize(archive_path)/1e6:.1f} MB)')
